# Agentic loop — direct to your vLLM endpoint

Sends the task brief + 4 tool schemas to your model, lets it decide what to call,
executes each call for real against `tools.py`, feeds the result back, stops when
it returns a final summary instead of a tool call. Edit the config cell and run.

In [1]:
# ============ CONFIG — edit, then run all ============
from dotenv import load_dotenv
import os
load_dotenv()

LOCAL = False

BASE_URL   = "http://localhost:8000/v1" if LOCAL else os.getenv("API_URL")
MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct" if LOCAL else os.getenv("API_MODEL_NAME")
API_KEY    = "not-needed" if LOCAL else os.getenv("RIT_API_KEY")

DATA_DIR = "../data"
BATCH_DIR = "c28_demo"

FLIGHT_DIR = f"{DATA_DIR}/{BATCH_DIR}/flights"            # directory of per-flight CSVs
METADATA   = f"{DATA_DIR}/{BATCH_DIR}/metadata.csv"
MODEL_PATH = f"{DATA_DIR}/c28_model_honest.joblib"       # trained via train_baseline.py

WORKDIR    = "../work/agent_run"                  # feats/preds/queue get written here
TOP_K      = 10
MAX_TURNS  = 12
# =======================================================

In [2]:
import sys, json
from pathlib import Path
import sys

note_dir = Path().resolve()
root = note_dir.parent

if str(root) not in sys.path:
    sys.path.append(str(root))

from scripts.agent_harness import run_agent, order_diff, VLLMClient

Path(WORKDIR).mkdir(parents=True, exist_ok=True)
FEATS = str(Path(WORKDIR) / "feats.csv")
PREDS = str(Path(WORKDIR) / "preds.json")
QUEUE = str(Path(WORKDIR) / "maintenance_queue.jsonl")

In [3]:
brief_path = Path("../openclaw/agent_task_brief.md")
SYSTEM_PROMPT = brief_path.read_text() if brief_path.exists() else (
    "You are the Orchestrator for a predictive-maintenance workflow. "
    "Call tools in order: inspect, featurize, classify, recommend. "
    "Stop after recommend with a HITL summary."
)

USER_GOAL = (
    f"Flight directory: {FLIGHT_DIR}\n"
    f"Metadata: {METADATA}\n"
    f"Model: {MODEL_PATH}\n"
    f"Feature output: {FEATS}\n"
    f"Predictions output: {PREDS}\n"
    f"Queue: {QUEUE}\n"
    f"Top-k: {TOP_K}\n"
    "Run the workflow and produce the HITL summary."
)

In [4]:
import json
holdout_auc = json.load(open("../data/c28_metrics_honest.json"))["holdout_roc_auc"]
USER_GOAL += f"\nModel holdout AUC (fold 0, never seen in training): {holdout_auc:.3f}"
f"\nModel holdout AUC (fold 0, never seen in training): {holdout_auc:.3f}"

'\nModel holdout AUC (fold 0, never seen in training): 0.739'

In [5]:
if LOCAL:
    client = VLLMClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY)
else:
    from scripts.agent_harness import OllamaClient
    client = OllamaClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY)


In [6]:
run = run_agent(client, SYSTEM_PROMPT, USER_GOAL, max_turns=MAX_TURNS, verbose=True)

[*] Warming up qwen3-coder:30b on the server (this may take a minute)...
[turn 1] inspect({'flight_dir': '../data/c28_demo/flights', 'metadata': '../data/c28_demo/metadata.csv'}) -> OK: {'tool': 'inspect_data', 'n_flights': 200, 'label_balance': {0: 107, 1: 93}, 'n_planes': 16, 'detected_channels': ['volt1', 'volt2', 'amp1', 'amp2', 'FQtyL', 'FQtyR', 'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM', 'E1 CHT1', 'E1 CHT2', 'E1 CHT3', 'E1 CHT4', 'E1 EGT1', 'E1 EGT2', 'E1 EGT3', 'E1 EGT4', 'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL'], 'n_channels': 23, 'example_flight_steps': 6363, 'notes': 'Read-only. Flags class balance and channel inventory for the planner.'}
[turn 2] featurize({'flight_dir': '../data/c28_demo/flights', 'metadata': '../data/c28_demo/metadata.csv', 'out': '../work/agent_run/feats.csv'}) -> OK: {'tool': 'featurize_flights', 'rows': 200, 'n_features': 277, 'out': '../work/agent_run/feats.csv'}
[turn 3] classify({'feats': '../work/agent_run/feats.csv', 'model': '../data/c28_model_ho

In [7]:
print("=== HITL SUMMARY ===")
print(run.final_summary)
print("\n=== stop reason ===")
print(run.stopped_reason)
print("\n=== call order ===")
print(json.dumps(order_diff(run), indent=2))

=== HITL SUMMARY ===
# HITL Summary

## Run scope
- **Flights inspected:** 200
- **Class balance:** 107 (no maintenance), 93 (maintenance needed)
- **Channels detected:** 23 (volt1, volt2, amp1, amp2, FQtyL, FQtyR, E1 FFlow, E1 OilT, E1 OilP, E1 RPM, E1 CHT1, E1 CHT2, E1 CHT3, E1 CHT4, E1 EGT1, E1 EGT2, E1 EGT3, E1 EGT4, OAT, IAS, VSpd, NormAc, AltMSL)

## Findings
- **HIGH urgency:** 56 flights
- **MEDIUM urgency:** 0 flights
- **LOW urgency:** 0 flights

## Top recommendations
1. **Flight:** flight_78.csv  
   **Probability:** 0.9743  
   **Urgency:** HIGH  
   **Action:** Schedule inspection within 2 days  

2. **Flight:** flight_1686.csv  
   **Probability:** 0.9714  
   **Urgency:** HIGH  
   **Action:** Schedule inspection within 2 days  

3. **Flight:** flight_1240.csv  
   **Probability:** 0.9707  
   **Urgency:** HIGH  
   **Action:** Schedule inspection within 2 days  

4. **Flight:** flight_2113.csv  
   **Probability:** 0.9598  
   **Urgency:** HIGH  
   **Action:** Schedul

In [10]:
# full trace
for t in run.trace:
    print(f"\nModel Name: {MODEL_NAME} | Local: {LOCAL} | Base URL: {BASE_URL}")
    print(f"\n{t.name}  ok={t.ok}")
    print("  args  :", t.args)
    print("  result:", json.dumps(t.result, indent=2)[:600])


Model Name: qwen3-coder:30b | Local: False | Base URL: https://api.genai.gccis.rit.edu/v1

inspect  ok=True
  args  : {'flight_dir': '../data/c28_demo/flights', 'metadata': '../data/c28_demo/metadata.csv'}
  result: {
  "tool": "inspect_data",
  "n_flights": 200,
  "label_balance": {
    "0": 107,
    "1": 93
  },
  "n_planes": 16,
  "detected_channels": [
    "volt1",
    "volt2",
    "amp1",
    "amp2",
    "FQtyL",
    "FQtyR",
    "E1 FFlow",
    "E1 OilT",
    "E1 OilP",
    "E1 RPM",
    "E1 CHT1",
    "E1 CHT2",
    "E1 CHT3",
    "E1 CHT4",
    "E1 EGT1",
    "E1 EGT2",
    "E1 EGT3",
    "E1 EGT4",
    "OAT",
    "IAS",
    "VSpd",
    "NormAc",
    "AltMSL"
  ],
  "n_channels": 23,
  "example_flight_steps": 6363,
  "notes": "Read-only. Flags class balance and channel inventory for the planner."


Model Name: qwen3-coder:30b | Local: False | Base URL: https://api.genai.gccis.rit.edu/v1

featurize  ok=True
  args  : {'flight_dir': '../data/c28_demo/flights', 'metadata': '../da